# Car Breakdown – Exploratory Data Analysis (EDA)

Doel: **begrijpen wanneer/waarom `breakdown_next_30_days = 1`** (auto gaat kapot binnen 30 dagen) en welke variabelen het meest bijdragen, zodat je straks beter kan **voorspellen**.

We volgen de **OSEMI** aanpak uit de cursus:
1. **Obtain**: data inlezen  
2. **Scrub**: datatypes / missing values / checks  
3. **Explore**: univariate + bivariate analyses, correlaties, segmentatie (merken, mileage, …)  
4. **Interpret**: “wat betekent dit?” + hypotheses voor het predictiemodel


In [ ]:
# 1) Obtain: libraries + data inlezen
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 10)

CSV_PATH = r"/mnt/data/train_CarBreakDown.csv"
df = pd.read_csv(CSV_PATH)

df.head(5)


In [ ]:
# basisinfo (zoals in de cursus)
df.shape, df.info()


## Snelle sanity checks

We kijken of:
- de targetkolom bestaat en enkel 0/1 bevat
- er duplicaten zijn
- er missing values zijn


In [ ]:
target = "breakdown_next_30_days"
df[target].value_counts(dropna=False)


In [ ]:
# duplicaten + missing values
dup_count = df.duplicated().sum()
nulls = df.isnull().sum().sort_values(ascending=False)

dup_count, nulls.head(10)


## 2) Scrub: datatypes & eenvoudige cleaning

We houden het exam-proof:
- kolomnamen netjes (lowercase is ok, maar hier laten we ze omdat ze al ok zijn)
- categorische kolommen als `category` (handig voor groupby/value_counts)
- zorgen dat numerieke kolommen numeriek zijn


In [ ]:
# categorische kolommen herkennen (object) en omzetten naar category
cat_cols = df.select_dtypes(include="object").columns.tolist()
cat_cols


In [ ]:
for c in cat_cols:
    df[c] = df[c].astype("category")

df.dtypes


## 3) Explore: Hoe vaak gaat een auto kapot? (baseline)

Eerst: percentage breakdowns. Dit is je **baseline**: als je altijd “geen breakdown” voorspelt, hoe goed scoor je dan?


In [ ]:
breakdown_rate = df[target].mean()
breakdown_rate


In [ ]:
# eenvoudige barplot van de target (aantallen)
counts = df[target].value_counts().sort_index()

plt.figure(figsize=(5,3))
plt.bar(counts.index.astype(str), counts.values)
plt.title("Aantal auto's per breakdown label")
plt.xlabel("breakdown_next_30_days")
plt.ylabel("count")
plt.grid(True, axis="y")
plt.show()


## Univariate: verdelingen van belangrijke numerieke variabelen

We starten met klassiekers die vaak “predictive” zijn:
- `vehicle_age_years`
- `mileage_km`
- `last_service_km_ago`
- `oil_quality_pct`
- `engine_hours`

We bekijken: **describe()** + **histogram** + **boxplot**.


In [ ]:
num_cols = [
    "vehicle_age_years", "mileage_km", "engine_hours", "last_service_km_ago",
    "oil_quality_pct", "avg_trip_length_km", "cleanliness_score",
    "driver_satisfaction_score"
]

df[num_cols].describe()


In [ ]:
# histograms (1 per keer om overzicht te houden)
col = "mileage_km"
plt.figure(figsize=(7,3))
plt.hist(df[col].dropna(), bins=30, ec="black")
plt.title(f"Histogram: {col}")
plt.xlabel(col)
plt.ylabel("count")
plt.grid(True)
plt.show()


In [ ]:
# boxplot (handig om outliers te zien)
col = "mileage_km"
plt.figure(figsize=(7,2.5))
plt.boxplot(df[col].dropna(), vert=False)
plt.title(f"Boxplot: {col}")
plt.xlabel(col)
plt.grid(True, axis="x")
plt.show()


## Bivariate: breakdown-rate per merk (vehicle_brand)

Dit beantwoordt direct jouw voorbeeld: “Fords gaan sneller kapot”.  
We berekenen per merk:
- aantal wagens
- breakdown-rate (= gemiddeld van 0/1)
We sorteren dalend op breakdown-rate.


In [ ]:
brand_stats = (
    df.groupby("vehicle_brand")[target]
      .agg(["count", "mean"])
      .rename(columns={"count":"n", "mean":"breakdown_rate"})
      .sort_values("breakdown_rate", ascending=False)
)

brand_stats.head(15)


In [ ]:
# Plot top 10 brands (met voldoende records)
min_n = 20
top = brand_stats[brand_stats["n"] >= min_n].head(10)

plt.figure(figsize=(9,4))
plt.bar(top.index.astype(str), top["breakdown_rate"].values)
plt.title(f"Top 10 merken met hoogste breakdown-rate (n >= {min_n})")
plt.xlabel("vehicle_brand")
plt.ylabel("breakdown_rate")
plt.xticks(rotation=45, ha="right")
plt.grid(True, axis="y")
plt.show()


## Mileage bins: “meer dan 200.000 km”

We maken een binned kolom voor `mileage_km` zodat je duidelijke drempels krijgt (exam-style interpretatie).
Daarna:
- breakdown-rate per mileage-bin
- breakdown-rate per (brand × mileage-bin) om jouw combo te vinden


In [ ]:
# mileage bins (pas gerust aan)
bins = [0, 50000, 100000, 150000, 200000, 250000, 300000, np.inf]
labels = ["0-50k","50-100k","100-150k","150-200k","200-250k","250-300k","300k+"]
df["mileage_bin"] = pd.cut(df["mileage_km"], bins=bins, labels=labels, right=False)

mileage_stats = df.groupby("mileage_bin")[target].agg(["count","mean"]).rename(columns={"count":"n","mean":"breakdown_rate"})
mileage_stats


In [ ]:
plt.figure(figsize=(8,4))
plt.bar(mileage_stats.index.astype(str), mileage_stats["breakdown_rate"].values)
plt.title("Breakdown-rate per mileage-bin")
plt.xlabel("mileage_bin")
plt.ylabel("breakdown_rate")
plt.xticks(rotation=30, ha="right")
plt.grid(True, axis="y")
plt.show()


In [ ]:
# combinatie merk x mileage_bin (zoals jouw voorbeeld)
combo = (
    df.groupby(["vehicle_brand","mileage_bin"])[target]
      .agg(["count","mean"])
      .rename(columns={"count":"n","mean":"breakdown_rate"})
      .reset_index()
)

# filter: genoeg data
combo_f = combo[combo["n"] >= 10].sort_values("breakdown_rate", ascending=False)
combo_f.head(20)


## Service & oliekwaliteit: duidelijke risicofactoren?

Hypothese (logisch):  
- **meer km sinds laatste service** → hogere breakdown-rate  
- **lagere oil_quality_pct** → hogere breakdown-rate

We bekijken dit met bins (gemakkelijk te interpreteren).


In [ ]:
# last_service_km_ago bins
svc_bins = [0, 2000, 5000, 10000, 20000, np.inf]
svc_labels = ["0-2k","2-5k","5-10k","10-20k","20k+"]
df["service_bin"] = pd.cut(df["last_service_km_ago"], bins=svc_bins, labels=svc_labels, right=False)

service_stats = df.groupby("service_bin")[target].agg(["count","mean"]).rename(columns={"count":"n","mean":"breakdown_rate"})
service_stats


In [ ]:
plt.figure(figsize=(8,4))
plt.bar(service_stats.index.astype(str), service_stats["breakdown_rate"].values)
plt.title("Breakdown-rate per 'km sinds laatste service' bin")
plt.xlabel("service_bin")
plt.ylabel("breakdown_rate")
plt.xticks(rotation=30, ha="right")
plt.grid(True, axis="y")
plt.show()


In [ ]:
# oil_quality_pct bins
oil_bins = [0, 20, 40, 60, 80, 100.1]
oil_labels = ["0-20","20-40","40-60","60-80","80-100"]
df["oil_bin"] = pd.cut(df["oil_quality_pct"], bins=oil_bins, labels=oil_labels, right=False)

oil_stats = df.groupby("oil_bin")[target].agg(["count","mean"]).rename(columns={"count":"n","mean":"breakdown_rate"})
oil_stats


In [ ]:
plt.figure(figsize=(8,4))
plt.bar(oil_stats.index.astype(str), oil_stats["breakdown_rate"].values)
plt.title("Breakdown-rate per oil_quality_pct bin")
plt.xlabel("oil_bin")
plt.ylabel("breakdown_rate")
plt.xticks(rotation=30, ha="right")
plt.grid(True, axis="y")
plt.show()


## Categorieën: fuel_type, tyre_type, weather_exposure

Voor categorische variabelen is dit exam-standaard:
- value_counts() (hoe vaak komt het voor?)
- breakdown-rate per categorie


In [ ]:
cat_interest = ["fuel_type", "tyre_type", "weather_exposure"]

cat_tables = {}
for c in cat_interest:
    tbl = df.groupby(c)[target].agg(["count","mean"]).rename(columns={"count":"n","mean":"breakdown_rate"}).sort_values("breakdown_rate", ascending=False)
    cat_tables[c] = tbl

cat_tables["fuel_type"].head(20)


In [ ]:
# voorbeeld plot voor fuel_type
tbl = cat_tables["fuel_type"]
plt.figure(figsize=(7,4))
plt.bar(tbl.index.astype(str), tbl["breakdown_rate"].values)
plt.title("Breakdown-rate per fuel_type")
plt.xlabel("fuel_type")
plt.ylabel("breakdown_rate")
plt.xticks(rotation=30, ha="right")
plt.grid(True, axis="y")
plt.show()


## Correlaties (alleen numeriek)

Zoals in de cursus: correlatie ≠ causatie, maar het helpt om relaties te spotten.


In [ ]:
num_df = df[num_cols + [target]].copy()
corr = num_df.corr(numeric_only=True)
corr


In [ ]:
# simpele heatmap via imshow (geen seaborn)
plt.figure(figsize=(9,7))
plt.imshow(corr.values, aspect="auto")
plt.title("Correlation matrix (numeric columns)")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.colorbar()
plt.tight_layout()
plt.show()


## “Wanneer gaat de auto het meeste kapot?” – Top segmenten

We definiëren een **segment** als combinatie van enkele sterke factoren (voorbeeld):
- hoge mileage (>= 200k)
- oude auto (>= 8 jaar)  *(drempel kan je aanpassen)*
- merk

We maken een tabel met breakdown-rate per segment en tonen de top.


In [ ]:
# drempels (pas aan indien nodig)
df["is_high_mileage"] = df["mileage_km"] >= 200_000
df["is_old"] = df["vehicle_age_years"] >= 8

seg = (
    df.groupby(["vehicle_brand","is_high_mileage","is_old"])[target]
      .agg(["count","mean"])
      .rename(columns={"count":"n","mean":"breakdown_rate"})
      .reset_index()
)

# focus op segmenten met genoeg data
seg_f = seg[seg["n"] >= 10].sort_values("breakdown_rate", ascending=False)
seg_f.head(25)


## 4) Interpret: wat nemen we mee naar het predictiemodel?

Schrijf (kort) jouw **findings** en **hypotheses** uit.

Voorbeelden die je in je verslag kan zetten:
- *Finding:* “Merken A/B hebben hogere breakdown-rate dan gemiddeld.”  
- *Finding:* “Breakdown-rate stijgt vanaf ~200k km.”  
- *Hypothese:* “Slechte oil_quality + veel km sinds service veroorzaakt meer breakdown.”  
- *Hypothese:* “Oude + high-mileage auto’s in bepaalde merken vormen het hoogste risico.”  

➡️ In de volgende stap (model) ga je dit testen met bv. logistic regression / tree-based model.


In [ ]:
# Extra: snelle tabel met overall baseline en een paar highlights
summary = {
    "rows": df.shape[0],
    "cols": df.shape[1],
    "baseline_breakdown_rate": df[target].mean(),
    "top_brands_by_rate (n>=20)": brand_stats[brand_stats["n"]>=20].head(5).to_dict(),
    "mileage_bin_rates": mileage_stats["breakdown_rate"].to_dict(),
    "service_bin_rates": service_stats["breakdown_rate"].to_dict(),
    "oil_bin_rates": oil_stats["breakdown_rate"].to_dict(),
}
summary
